In [1]:
using LogDensityProblems: LogDensityProblems;
using Distributions
using DelimitedFiles
using Random: Random
using DynamicPPL

In [2]:
using Turing

In [4]:
n_schools = 8
y = [28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0] # estimated treatment effects
σ = [15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0]
@model function school_reparam(y, σ, n_schools = 8)
	μ ~ Normal(0, 5)
	τ ~ truncated(Cauchy(0, 5), lower = 0)
	θ ~ filldist(Normal(0, 1), n_schools)
	for i in 1:n_schools
		y[i] ~ Normal(μ + τ * θ[i], σ[i])
	end
end

school_reparam (generic function with 4 methods)

In [9]:
# Let's define some type that represents the model.
struct RegressionProblem{Ty <: AbstractVector}
	y::Ty
	σ::Ty
end
LogDensityProblems.dimension(model::RegressionProblem) = 10

function LogDensityProblems.logdensity(model::RegressionProblem, parameters::AbstractVector{<:Real})
	μ,τ,θ = parameters[1],parameters[2],parameters[3:end]
	lp = logpdf(Normal(0, 5), μ)
	lp += logpdf(truncated(Cauchy(0, 5),lower=0), τ)

	for i in 1:8
		lp += logpdf(Normal(0, 1), θ[i])
		lp += logpdf(Normal(μ + τ * θ[i], model.σ[i]), model.y[i])
	end
	return lp
end

LogDensityProblems.capabilities(model::RegressionProblem) = LogDensityProblems.LogDensityOrder{0}()

In [10]:
MYmodel = RegressionProblem(y, σ)

RegressionProblem{Vector{Float64}}([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0], [15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])

In [24]:
m = AbstractMCMC.LogDensityModel(MYmodel).logdensity
using Random
rng = MersenneTwister(1234)
LogDensityProblems.logdensity(m,rand(rng,10))

-44.22549218755314

In [25]:
function tune_lengthscale(t, μ, N_e, N_c, M_adapt)
	N_e = max(1, N_e)

	if t <= M_adapt
		return 2μ * N_e / (N_e + N_c)
	else
		return μ
	end
end

function get_complementary(i, N)
	indices = collect(1:N)
	deleteat!(indices, i)
	return indices
end

function get_direction_vector(S, l, m, μ)
	return μ * (S[l].x - S[m].x)
end

""" 
	DifferentialMove(rng, k, μ, S, N)

	Perform a differential move for walker k.

	# Arguments
	- `rng::AbstractRNG`: Random number generator.
	- `k::Int`: Index of the walker.
	- `μ::Float64`: Lengthscale.
	- `S::Array{Float64, 2}`: Array of walker positions.
	- `N::Int`: Number of walkers.

	# Returns
	- `ηₖ::Array{Float64, 1}`: Differential move.
"""
function DifferentialMove(rng, k, μ, S, N)
	# work on walker k
	indices = get_complementary(k, N)
	# draw two random indices from the complementary set, without replacement
	l, m = sample(rng, indices, 2, replace = false)
	return get_direction_vector(S, l, m, μ)
end

DifferentialMove

In [26]:
using AbstractMCMC

struct EnsembleSliceSampler{T<:Float64,A<:Int64} <: AbstractMCMC.AbstractSampler
    "initial length scale"
    μ_init::T
    "number of adapation steps"
    M_adapt::A
    "number of walkers"
    N_walkers::A
    "max number of attempts"
    max_steps::A
end
struct Walker{A<:AbstractVector{<:Real}}
    "current position"
    x::A # a matrix of dimension n_params
end

struct ESState{A<:AbstractVector,T<:Float64,B<:Int64}
    # "current position"
    # vi::V
    "current position"
    x::A
    "length scale"
    μ::T
    "iteration"
    t::B
end

struct ESSample{A<:AbstractVector}
    "current position"
    x::A # a matrix of dimension n_walkers * n_params
end

In [32]:
function AbstractMCMC.step(
	rng::Random.AbstractRNG,
	model_wrapper::AbstractMCMC.LogDensityModel,
	sampler::EnsembleSliceSampler,
	state::ESState)

	model = model_wrapper.logdensity
	# extract the sampler parameters
	μ = sampler.μ_init
	M_adapt = sampler.M_adapt
	N_walkers = sampler.N_walkers
	max_steps = sampler.max_steps

	f(y) = LogDensityProblems.logdensity(model, y)

	# extract current state
	S, μ, t = state.x, state.μ, state.t
	N_dim = size(S, 2)

	x_new = Vector(undef, N_walkers)
	# Matrix{Float64}(undef, N_walkers, N_dim)

	R, L, N_e, N_c = 0, 0, 0, 0
	X′ = 0

	# loop over the walkers
	for k in 1:N_walkers
		Xₖ = S[k].x # get the current position of walker k
		ηₖ = DifferentialMove(rng, k, μ, S, N_walkers) # get the differential move

		δ = rand(rng, Exponential(1))
		Y = f(Xₖ) - δ

		L = -rand(rng)
		R = L + 1
		l = 0
		while Y < f(L .* ηₖ + Xₖ)
			L = L - 1
			N_e = N_e + 1
			l += 1
			if l == max_steps
				println("L: ", L, " Y: ", Y, " f(L): ", f(L .* ηₖ + Xₖ))
				error("Max steps reached", " iteration: ", t, " walker: ", k)
			end
		end
		l = 0
		while Y < f(R .* ηₖ + Xₖ)
			R = R + 1
			N_e = N_e + 1
			l += 1
			if l == max_steps
				println("L: ", R, " Y: ", Y, " f(R): ", f(R .* ηₖ + Xₖ))
				error("Max steps reached")
			end
		end

		l = 0
		while true
			l += 1
			X′ = rand(rng, Uniform(L, R))
			Y′ = f(X′ .* ηₖ + Xₖ)
			if Y < Y′
				break
			end
			if X′ < 0
				L = X′
				N_c = N_c + 1
			else
				R = X′
				N_c = N_c + 1
			end
			if l == max_steps
				println("L: ", R, " Y: ", Y, " f(R): ", f(R .* ηₖ + Xₖ))

				error("Max steps reached")
			end
		end

		Xₖ = X′ .* ηₖ + Xₖ
		x_new[k] = Walker(Xₖ)
		# push!(x_new, Walker(Xₖ))
	end
	# println("R: ", R, " L: ", L, " N_e: ", N_e, " N_c: ", N_c, " μ: ", μ)
	μ = tune_lengthscale(t, μ, N_e, N_c, M_adapt)
	t += 1
	# println((x_new) isa Vector{<:Walker})
	state_new = ESState(x_new, μ, t)
	return ESSample(x_new), state_new
end

In [33]:
rng = Random.default_rng(89)
ndims = 10
nwalkers = 2 * ndims
sampler = EnsembleSliceSampler(1.0, 100, 6, 10_000)
init_p = [ Walker(randn(rng,ndims)) for i in 1:nwalkers]
state = ESState(init_p, 1.0, 1)

ESState{Vector{Walker{Vector{Float64}}}, Float64, Int64}(Walker{Vector{Float64}}[Walker{Vector{Float64}}([-1.1926362660983154, -0.29707315075930657, -0.3669020282733443, -0.18278265435950258, 0.272369416699064, 1.2856428534606623, -1.4654868163175985, 1.1256386168328416, -0.23615713510395403, 0.7712860995691628]), Walker{Vector{Float64}}([-1.1379642122962674, 0.3011207899232353, -0.8600022275493345, 0.5698426695770056, -2.152102743164318, 1.3644741838945718, 0.11467451356270833, 2.294886963986033, 0.9671684959553689, 0.8664932522192766]), Walker{Vector{Float64}}([0.7777198469786676, -0.10058523933146352, 2.702825981718758, -0.4473210045522744, -1.116591205863033, -0.5681187637138319, -1.1297224779845925, 0.275328190262871, 0.4901756926109389, 0.08051888231905702]), Walker{Vector{Float64}}([0.9350626574131896, 1.2259051539728734, -1.7388700933132553, -0.8760665789974732, -1.0043329085013752, -1.828572710271645, -2.143526071919035, 1.070369659034976, -0.939536696679879, 0.516997813508766

In [34]:
rng = Random.default_rng(132)

x_next, state_next = AbstractMCMC.step(
    rng,
    AbstractMCMC.LogDensityModel(MYmodel),
    sampler,
    state
)

L: -10000.485238819852 Y: -Inf f(L): -3.227208421941113e15


ErrorException: Max steps reached iteration: 1 walker: 1

In [35]:
using DynamicPPL

In [83]:
function AbstractMCMC.step(
    rng::Random.AbstractRNG,
    model_wrapper::AbstractMCMC.LogDensityModel,
    ::EnsembleSliceSampler;
    kwargs...)
    model = model_wrapper.logdensity
    nwalkers = sampler.N_walkers
    ndims = LogDensityProblems.dimension(model)
    # x = randn(rng,nwalkers,ndims)
    init_p = [ Walker(randn(rng,ndims)) for i in 1:nwalkers]
    # x = [DynamicPPL.VarInfo(rng, model, DynamicPPL.SampleFromPrior()) for _ in 1:nwalkers]
    # x = mapreduce(permutedims, hcat, x)
    return ESSample(init_p),  ESState(init_p, 1.0, 1)
end

In [87]:

# samples = sample(MYmodel, sampler, 1_000; initial_state=state, progress=false)
# using MCMCChains
# samples_matrix = stack(sample -> sample.x, samples)
# samples_matrix = stack(sample -> sample.x, samples_matrix)
# samples_matrix = permutedims(samples_matrix, [3, 1, 2])

# ch = Chains(samples_matrix,[:α,:β,:γ])
# using StatsPlots
# plot(ch[100:end,:,:])

In [90]:
using LinearAlgebra

@model function InferenceModel(y, t)
	α ~ Normal(2, 1)
	β ~ Normal(0, 1)
	γ ~ Normal(-2, 1)
	return y ~ MvNormal(fun(t, α, β, γ), I)
end

mymod = InferenceModel(y, t)
# Turing.Inference.getparams(::Turing.Model, sample::ESSample) = sample.x
function Turing.Inference.getparams(::Turing.Model, sample::ESSample) 
	# for walker in sample.x
	# 	println(walker.x)
	# end
	return [sample.x[i].x for i in 1:length(sample.x)]
end

In [91]:
vi = DynamicPPL.VarInfo(rng, mymod, DynamicPPL.SampleFromPrior()) 

TypedVarInfo{@NamedTuple{α::DynamicPPL.Metadata{Dict{VarName{:α, typeof(identity)}, Int64}, Vector{Normal{Float64}}, Vector{VarName{:α, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}, β::DynamicPPL.Metadata{Dict{VarName{:β, typeof(identity)}, Int64}, Vector{Normal{Float64}}, Vector{VarName{:β, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}, γ::DynamicPPL.Metadata{Dict{VarName{:γ, typeof(identity)}, Int64}, Vector{Normal{Float64}}, Vector{VarName{:γ, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}}, Float64}((α = DynamicPPL.Metadata{Dict{VarName{:α, typeof(identity)}, Int64}, Vector{Normal{Float64}}, Vector{VarName{:α, typeof(identity)}}, Vector{Float64}, Vector{Set{DynamicPPL.Selector}}}(Dict(α => 1), [α], UnitRange{Int64}[1:1], [0.4993117530487674], Normal{Float64}[Normal{Float64}(μ=2.0, σ=1.0)], Set{DynamicPPL.Selector}[Set()], [0], Dict{String, BitVector}("del" => [0], "trans" => [0])), β = DynamicPPL.Metadata{Di

In [93]:
Turing.Inference.Transition

Turing.Inference.Transition

In [73]:
# sampler = Emcee(50)

In [74]:
# chain = sample(mymod, sampler, 10_00)

In [92]:
chain = sample(mymod, externalsampler(sampler), 100)

Sampling   0%|                                          |  ETA: N/A
Sampling 100%|██████████████████████████████████████████| Time: 0:00:00


MethodError: MethodError: no method matching DynamicPPL.Metadata(::Dict{VarName{:α, typeof(identity)}, Int64}, ::Vector{VarName{:α, typeof(identity)}}, ::Vector{UnitRange{Int64}}, ::Vector{Vector{Float64}}, ::Vector{Normal{Float64}}, ::Vector{Set{DynamicPPL.Selector}}, ::Vector{Int64}, ::Dict{String, BitVector})

Closest candidates are:
  DynamicPPL.Metadata(::TIdcs, ::TVN, ::Vector{UnitRange{Int64}}, !Matched::TVal, ::TDists, ::TGIds, ::Vector{Int64}, ::Dict{String, BitVector}) where {TIdcs<:(Dict{<:VarName, Int64}), TDists<:(AbstractVector{<:Distribution}), TVN<:(AbstractVector{<:VarName}), TVal<:(AbstractVector{<:Real}), TGIds<:AbstractVector{Set{DynamicPPL.Selector}}}
   @ DynamicPPL ~/.julia/packages/DynamicPPL/DvdZw/src/varinfo.jl:47


In [ ]:
samples_matrix = stack(sample -> sample.x, samples)
ch = Chains(permutedims(samples_matrix, [3, 2, 1]),[:α,:β,:γ])

In [ ]:
ch[100:end,:,:]

In [ ]:
using StatsPlots
plot(ch[100:end,:,:])

In [131]:
# using DelimitedFiles
# data = readdlm("data.txt")
# t, y, yerr = data[:, 1], data[:, 2], data[:, 3]


# Turing.Inference.getparams(::Turing.Model, sample::ESSample) = sample.x

# fun(x,α,β,γ) = @. α * x^2 + β * x + γ
# x = 0.0:0.5:15.0

# @model function InferenceModel(y)
# 	α ~ Normal(2, 1)
# 	β ~ Normal(0, 1)
# 	γ ~ Normal(-2, 1)

# 	y ~ MvNormal(fun(x, α, β, γ), 1)
# end

# m = InferenceModel(y)
# m2 = m | (α=1.0, β=0.3, γ=-2.)

In [132]:
# DynamicPPL.SampleFromPrior()